In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## No memory

In [4]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [14]:
import requests, os
from dotenv import load_dotenv
load_dotenv()

response = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}"}
)

models = response.json()["data"]
for m in sorted(models, key=lambda x: x["id"]):
    print(m["id"])

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
groq/compound
groq/compound-mini
llama-3.1-8b-instant
llama-3.3-70b-versatile
meta-llama/llama-4-scout-17b-16e-instruct
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3-32b
whisper-large-v3
whisper-large-v3-turbo


In [17]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# Replace with any model ID from the list above
groq_model = ChatGroq(model="llama-3.1-8b-instant")  

agent = create_agent(
    model=groq_model,
    tools=[web_search]
)

In [18]:
from langchain_core.messages import HumanMessage


question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

Nice to meet you Seán. However, I don't have any information about you. Would you like to know something about the colour green or maybe something else?


In [19]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='129816e5-2c73-4475-a0b7-e64351e32052'),
              AIMessage(content="Nice to meet you Seán. However, I don't have any information about you. Would you like to know something about the colour green or maybe something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 222, 'total_tokens': 256, 'completion_time': 0.649694268, 'completion_tokens_details': None, 'prompt_time': 0.022756738, 'prompt_tokens_details': None, 'queue_time': 0.688627279, 'total_time': 0.672451006}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfd5-3b17-74c3-aa26-c2efc6678879-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 34, 

In [20]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response)

{'messages': [HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='50923896-6cca-4c54-9539-60f1c8df30db'),
              AIMessage(content="Unfortunately, I don't have any information about your personal preferences.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 216, 'total_tokens': 230, 'completion_time': 0.022931014, 'completion_tokens_details': None, 'prompt_time': 0.012259364, 'prompt_tokens_details': None, 'queue_time': 0.157467674, 'total_time': 0.035190378}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfd5-73f6-7111-b994-6ee6c93d3a2a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 216, 'output_tokens': 14, 'total_tokens': 230})]}


## Memory

In [21]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

groq_model = ChatGroq(model="llama-3.1-8b-instant")  # use whichever model worked for you

agent = create_agent(
    model=groq_model,
    tools=[],
    checkpointer=InMemorySaver()
)

In [22]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [23]:
pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='65d00a1e-cf21-4a6c-b196-2bae742e7e10'),
              AIMessage(content='Nice to meet you Seán.  Green is a wonderful colour, very calming and natural. What do you like most about the colour green?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 47, 'total_tokens': 77, 'completion_time': 0.054407668, 'completion_tokens_details': None, 'prompt_time': 0.013869199, 'prompt_tokens_details': None, 'queue_time': 0.419784234, 'total_time': 0.068276867}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfd6-96be-7411-bda3-e13f7a7d37bb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 30, 'total_tokens': 77})]}


In [24]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='65d00a1e-cf21-4a6c-b196-2bae742e7e10'),
              AIMessage(content='Nice to meet you Seán.  Green is a wonderful colour, very calming and natural. What do you like most about the colour green?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 47, 'total_tokens': 77, 'completion_time': 0.054407668, 'completion_tokens_details': None, 'prompt_time': 0.013869199, 'prompt_tokens_details': None, 'queue_time': 0.419784234, 'total_time': 0.068276867}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebfd6-96be-7411-bda3-e13f7a7d37bb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 30, 'total_tokens': 77}),
         